In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt

In [2]:
# Time series plotting for 2D aerosol data
def timeseries(file, lat_lims, lon_lims, method, start, end, aerosol):
    '''
    Plots time series of selected pollutant over desired latitude/longitude range and time range using chosen aggregation method. Plots each time series in its own figure window.
    Takes arguments (file='path_to_file', lat_lims=[left=value, right=value], lon_lims=[left=value, right=value], method='mean'/'sum'/'median', start='YYYY-MM-DD' (datetime form of desired start time), 
    end='YYYY-MM-DD' (datetime form of desired end time), aerosol='aerosol_name'). Meant to read in netcdf files of emissions from CESM2.2 model runs. Files should have latitude, longitude, and time coordinates.
    '''
    if lat_lims[0] >= lat_lims[1]:
        raise SyntaxError('First latitude input must be smaller than the second.')

    if lon_lims[0] >= lon_lims[1]:
        raise SyntaxError('First longitude input must be smaller than the second.')
        
    ds = xr.open_dataset(file)
    area = ds.sel(lat=slice(lat_lims[0], lat_lims[1]))
    area = area.sel(lon=slice(lon_lims[0], lon_lims[1]))
    
    if method == 'mean':
        result = area.mean(dim=['lat','lon'])
    elif method == 'sum':
        result = area.sum(dim=['lat','lon'])
    elif method == 'median':
        result = area.median(dim=['lat','lon'])
    else:
        raise ValueError('Not a supported aggregation method. Choose either mean, median, or sum.')
        
    result = result.drop_vars('date')
    values = result.sel(time=slice(start, end))
    values_arr = values.to_dataarray()
    
    fig = plt.figure()
    ax = plt.axes()
    values_arr.plot() 
    method = method.capitalize()
    aerosol = aerosol.capitalize()
    plt.title(f'{method} {aerosol} from {lat_lims[0]}-{lat_lims[1]} Degrees North and {lon_lims[0]}-{lon_lims[1]} Degrees East Over Time')
    ax.set_xlabel('Time')
    ax.set_ylabel(f'{aerosol} Emission Flux (kgm⁻²s⁻¹)')
    
    return None

In [3]:
# Plot multiple aerosol time series on a single plot
def timeseries_compare(files, lat_lims, lon_lims, method, start, end, aerosols):
    '''Plots time series of multiple pollutants over a selected lat/lon range and tine range on a single plot. Takes arguments (files=iterable_coll_of_files, lat_lims=[left=value, right=value],
       lon_lims=[left=value, right=value], method='mean'/'sum'/'median', start='YYYY-MM-DD' (datetime form of desired start time), end='YYYY-MM-DD' (datetime form of desired end time),
       aerosols=iterable_of_aerosol_names). Meant to read in netcdf files of emissions from CESM2.2 model runs. Files should have latitude, longitude, and time coordinates. '''
    
    # Error handling
    if len(files) == len(aerosols):
        pass
    else:
        raise SyntaxError('Number of data files and aerosol names must be equal.')

    if lat_lims[0] >= lat_lims[1]:
        raise SyntaxError('First latitude input must be smaller than the second.')

    if lon_lims[0] >= lon_lims[1]:
        raise SyntaxError('First longitude input must be smaller than the second.')
        
    for i in range(len(aerosols)):
        aerosols[i] = (aerosols[i]).capitalize()
    method = method.capitalize()
    fig = plt.figure()
    ax = plt.axes()

    lines = []
    
    for f in files:
        ds = xr.open_dataset(f)
        area = ds.sel(lat=slice(lat_lims[0], lat_lims[1]))
        area = area.sel(lon=slice(lon_lims[0], lon_lims[1]))
    
        if method == 'Mean':
            result = area.mean(dim=['lat','lon'])
        elif method == 'Sum':
            result = area.sum(dim=['lat','lon'])
        elif method == 'Median':
            result = area.median(dim=['lat','lon'])
        else:
            raise ValueError('Not a supported aggregation method. Choose either mean, median, or sum.')
        
        result = result.drop_vars('date')
        values = result.sel(time=slice(start, end))
        values_arr = values.to_dataarray()
        val = values_arr.plot()
        lines.append(val[0])

        
    ax.set_xlabel('Time')
    ax.set_ylabel('Emission Flux (kgm⁻²s⁻¹)')
    plt.title(f'{method} of {', '.join(a for a in aerosols)} from {lat_lims[0]}-{lat_lims[1]} Degrees North and {lon_lims[0]}-{lon_lims[1]} Degrees East Over Time')
    plt.legend(lines, aerosols)
    
    return None